# Shapley Methods — Cross-Dataset Summary

Aggregated metrics across all 4 synthetic datasets. Reuses helper functions from `analysis_utils.py`.

| Dataset | Confounders | Data type |
|---------|------------|-----------|
| `linear_conf_f50_s1000_p30` | Yes | Linear |
| `linear_no_conf_f50_s1000_p30` | No | Linear |
| `mixed_conf_f50_s1000_30` | Yes | Mixed |
| `mixed_no_conf_f50_s1000_p30` | No | Mixed |

**Metrics computed:**
- **GSS** — Graph Sensitivity Score (PC vs LiNGAM magnitude shift)
- **Sign Alignment** vs True graph — fraction of instances with matching sign
- **Magnitude TGA** vs True graph — relative magnitude deviation

In [1]:
import sys
import importlib
from pathlib import Path as _Path
sys.path.insert(0, str(_Path("..").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Force reload so any changes to analysis_utils.py are picked up
if "utils.analysis_utils" in sys.modules:
    importlib.reload(sys.modules["utils.analysis_utils"])

from utils.analysis_utils import (
    mean_abs_shap,
    compute_sign_alignment, compute_tga, compute_sss,
    compute_gss, top_k_features, rank_shift_top_k,
    flow_graph_stats,
    build_dag_graph, make_dag_pos, draw_dag_highlight,
    plot_gss_sss_scatter, plot_tga_sa_scatter,
)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR           = Path("..").resolve()
EXPLAINABILITY_DIR = BASE_DIR / "data" / "explainability"
CAUSAL_DIR         = BASE_DIR / "data" / "causal"
PROCESSED_DIR      = BASE_DIR / "data" / "processed"
SYNTHETIC_DIR      = BASE_DIR / "data" / "synthetic"

# ── Datasets to compare ───────────────────────────────────────────────────────
DATASETS = [
    "linear_conf_f50_s1000_p30",
    # "linear_no_conf_f50_s1000_p30",
    # "mixed_conf_f50_s1000_p30",
    # "mixed_no_conf_f50_s1000_p30",
]
MODEL = "lgbm"
BASE_METHODS = ["Asymmetric", "Causal", "Flow"]
DISC_GRAPHS  = ["PC", "LiNGAM"]

METHOD_COLORS = {
    "Scratch":    "#636363",
    "Asymmetric": "#2166ac",
    "Causal":     "#4dac26",
    "Flow":       "#d01c8b",
}
# Short dataset labels for display
DS_LABELS = {
    "linear_conf_f50_s1000_p30":     "Lin-Conf",
    "linear_no_conf_f50_s1000_p30":  "Lin-NoConf",
    "mixed_conf_f50_s1000_p30":      "Mix-Conf",
    "mixed_no_conf_f50_s1000_p30":   "Mix-NoConf",
}

print("Setup complete. Datasets:", DATASETS)

Setup complete. Datasets: ['linear_conf_f50_s1000_p30']


## 1. Load All Data

Load Shapley values, causal graphs, metadata, and pre-computed true-graph arrays for every dataset.

In [2]:

import joblib as _joblib

MODELS_DIR    = BASE_DIR / "models"
PROCESSED_DIR = BASE_DIR / "data" / "processed"


def _compute_output_std(dataset: str, model: str = "lgbm") -> float | None:
    """Load saved model + test split and return prediction std."""
    model_path = MODELS_DIR / f"{dataset}_{model}.pkl"
    test_path  = PROCESSED_DIR / f"{dataset}_test.parquet"
    if not model_path.exists() or not test_path.exists():
        return None
    model_data = _joblib.load(model_path)
    lgbm_model = model_data["model"]
    sel_feat   = model_data.get("selected_features")
    df         = pd.read_parquet(test_path)
    X          = df.drop(columns=["Y"], errors="ignore")
    if sel_feat is not None:
        X = X[sel_feat]
    preds = lgbm_model.predict(X)
    return float(preds.std())


def load_dataset(dataset: str, model: str = "lgbm") -> dict:
    """
    Load all Shapley values, causal graphs, metadata, and true-graph arrays
    for one dataset. Returns a dict with all data needed for metric computation.
    """
    exp_base = EXPLAINABILITY_DIR / dataset / model
    causal   = CAUSAL_DIR

    # ── Main SHAP data (from pipeline output) ─────────────────────────────
    keys = {
        "Scratch":             exp_base / "scratch"  / "shapley_values.npy",
        "Asymmetric (PC)":     exp_base / "pc"        / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"        / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"        / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
    }
    subset_keys = {
        "Asymmetric (PC)":     exp_base / "pc"        / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"        / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"        / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
        "Asymmetric (True)":   exp_base / "true"      / "asymmetric" / "shapley_values.npy",
        "Causal (True)":       exp_base / "true"      / "causal"     / "shapley_values.npy",
        "Flow (True)":         exp_base / "true"      / "flow"       / "shapley_values.npy",
    }

    shap_data        = {k: np.load(v) for k, v in keys.items()        if v.exists()}
    subset_shap_data = {k: np.load(v) for k, v in subset_keys.items() if v.exists()}

    # ── Feature names ──────────────────────────────────────────────────────
    with open(causal / f"{dataset}_pc_results.json") as f:
        pc_results = json.load(f)
    with open(causal / f"{dataset}_lingam_results.json") as f:
        lingam_results = json.load(f)
    feature_names = [n for n in pc_results["feature_names"] if n != "Y"]

    # ── True Y-parents ─────────────────────────────────────────────────────
    with open(SYNTHETIC_DIR / f"{dataset}_metadata.json") as f:
        meta = json.load(f)
    y_parent_set = set(meta["y_parent_indices"])

    # ── Graph stats ────────────────────────────────────────────────────────
    pc_edges,     pc_sources     = flow_graph_stats(pc_results["adjacency_matrix"],     len(feature_names))
    lingam_edges, lingam_sources = flow_graph_stats(lingam_results["adjacency_matrix"], len(feature_names))

    # ── Model output std (for metric normalisation) ──────────────────────
    output_std = _compute_output_std(dataset, model)

    return dict(
        shap_data        = shap_data,
        subset_shap_data = subset_shap_data,
        feature_names    = feature_names,
        y_parent_set     = y_parent_set,
        pc_edges         = pc_edges,
        lingam_edges     = lingam_edges,
        output_std       = output_std,
    )


# Load all datasets
all_data = {}
for ds in DATASETS:
    all_data[ds] = load_dataset(ds)
    n_shap = all_data[ds]["shap_data"]["Scratch"].shape[0]
    n_feat = len(all_data[ds]["feature_names"])
    n_par  = len(all_data[ds]["y_parent_set"])
    loaded = len(all_data[ds]["shap_data"])
    sub_loaded = len(all_data[ds]["subset_shap_data"])
    r = all_data[ds]["output_std"]
    print(f"  {DS_LABELS[ds]:12s}  {loaded} main SHAP + {sub_loaded} subset  "
          f"n_shap={n_shap}  n_feat={n_feat}  true_parents={n_par}  "
          f"output_std={r:.4f}" if r else
          f"  {DS_LABELS[ds]:12s}  {loaded} main SHAP + {sub_loaded} subset  "
          f"n_shap={n_shap}  n_feat={n_feat}  true_parents={n_par}  output_std=None")


  Lin-Conf      7 main SHAP + 9 subset  n_shap=100  n_feat=50  true_parents=15  output_std=4.7094


## 2. Compute All Metrics

For each dataset × method × graph, compute GSS, SSS, Sign Alignment, and Magnitude TGA.

In [3]:

gss_metrics = []   # one per (dataset, method)
sss_metrics = []   # one per (dataset, method)
tga_metrics = []   # one per (dataset, method, graph)
sa_metrics  = []   # sign alignment, one per (dataset, method, graph)
sa_scratch_metrics  = []   # sign alignment vs Scratch
tga_scratch_metrics = []   # TGA vs Scratch

# Store per-feature arrays by dataset for diagnostics and visualization
gss_per_feat = {}   # dict[dataset] -> dict[method] -> ndarray(n_features,)
sss_per_feat = {}   # dict[dataset] -> dict[method] -> ndarray(n_features,)

for ds in DATASETS:
    d             = all_data[ds]
    shap_data     = d["shap_data"]
    subset_shap   = d["subset_shap_data"]
    feature_names = d["feature_names"]
    output_std    = d.get("output_std")

    # ── GSS — RMS + output_std normalisation ─────────────────────────────
    gss_feat = compute_gss(shap_data, BASE_METHODS, output_std=output_std)
    gss_per_feat[ds] = gss_feat
    for meth, feat_arr in gss_feat.items():
        gss_metrics.append(dict(
            Dataset   = DS_LABELS[ds],
            Method    = meth,
            MeanGSS   = float(np.mean(feat_arr)),
            MedianGSS = float(np.median(feat_arr)),
        ))

    # ── SSS — sign stability (no normalisation needed) ────────────────────
    sss_feat = compute_sss(shap_data, BASE_METHODS)
    sss_per_feat[ds] = sss_feat
    for meth, feat_arr in sss_feat.items():
        sss_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            MeanSSS = float(np.nanmean(feat_arr)),
        ))

    # ── Sign Alignment vs True ─────────────────────────────────────────────
    sign_align = compute_sign_alignment(subset_shap, BASE_METHODS, DISC_GRAPHS, reference="True")
    for (meth, g), arr in sign_align.items():
        sa_metrics.append(dict(
            Dataset       = DS_LABELS[ds],
            Method        = meth,
            Graph         = g,
            MeanSignAlign = float(np.nanmean(arr)),
        ))

    # ── Magnitude TGA vs True — RMS + output_std normalisation ───────────
    tga_data, _ = compute_tga(
        subset_shap, feature_names, BASE_METHODS, DISC_GRAPHS,
        reference="True", output_std=output_std,
    )
    for (meth, g), arr in tga_data.items():
        tga_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            Graph   = g,
            MeanTGA = float(np.mean(arr)),
        ))

    # ── Sign Alignment vs Scratch (Traditional baseline) ──────────────────
    combined = dict(subset_shap)
    if "Scratch" in shap_data:
        combined["Scratch"] = shap_data["Scratch"]

    sign_align_scratch = compute_sign_alignment(
        combined, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
    )
    for (meth, g), arr in sign_align_scratch.items():
        sa_scratch_metrics.append(dict(
            Dataset       = DS_LABELS[ds],
            Method        = meth,
            Graph         = g,
            MeanSignAlign = float(np.nanmean(arr)),
        ))

    # ── Magnitude TGA vs Scratch — RMS + output_std normalisation ────────
    tga_scratch, _ = compute_tga(
        combined, feature_names, BASE_METHODS, DISC_GRAPHS,
        reference="Scratch", output_std=output_std,
    )
    for (meth, g), arr in tga_scratch.items():
        tga_scratch_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            Graph   = g,
            MeanTGA = float(np.nanmean(arr)),
        ))

# ── Build summary DataFrames ───────────────────────────────────────────────
df_gss         = pd.DataFrame(gss_metrics)
df_sss         = pd.DataFrame(sss_metrics)
df_sa          = pd.DataFrame(sa_metrics)
df_tga         = pd.DataFrame(tga_metrics)
df_sa_scratch  = pd.DataFrame(sa_scratch_metrics)
df_tga_scratch = pd.DataFrame(tga_scratch_metrics)

print(f"Metrics computed:")
print(f"  GSS: {len(df_gss)} rows (per-feature arrays stored in gss_per_feat)")
print(f"  SSS: {len(df_sss)} rows (per-feature arrays stored in sss_per_feat)")
print(f"  Sign Alignment vs True: {len(df_sa)} rows")
print(f"  TGA vs True: {len(df_tga)} rows")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")


Metrics computed:
  GSS: 3 rows (per-feature arrays stored in gss_per_feat)
  SSS: 3 rows (per-feature arrays stored in sss_per_feat)
  Sign Alignment vs True: 6 rows
  TGA vs True: 6 rows
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows


In [4]:

def feature_profile(feature_name: str, method_key: str, dataset: str = None):
    """
    Print a full metric profile for one feature under one method/graph combination
    and return a rank-comparison DataFrame across all loaded methods.

    Parameters
    ----------
    feature_name : str   Feature name, e.g. "X_5"
    method_key   : str   Method key, e.g. "Asymmetric (PC)", "Causal (LiNGAM)", "Scratch"
    dataset      : str   Dataset name from DATASETS, or None to loop over all datasets.

    Returns
    -------
    rank_df : pd.DataFrame
        Rank of this feature across all loaded method keys (last dataset if multiple).
    """
    # ── Parse base method and graph from key ─────────────────────────────────
    if " (" in method_key and method_key.endswith(")"):
        base_meth = method_key[: method_key.rfind(" (")]
        graph     = method_key[method_key.rfind("(") + 1 : -1]
    else:
        base_meth = method_key
        graph     = None

    ds_list  = [dataset] if dataset else DATASETS
    rank_df  = pd.DataFrame()
    W        = 72

    for ds in ds_list:
        d             = all_data[ds]
        shap_d        = d["shap_data"]
        subset_shap_d = d["subset_shap_data"]
        feat_names    = d["feature_names"]
        y_parent_set  = d["y_parent_set"]
        output_std    = d.get("output_std")
        n_feat        = len(feat_names)

        if feature_name not in feat_names:
            print(f"[{DS_LABELS[ds]}] Feature '{feature_name}' not found. "
                  f"Available: {feat_names[:5]} ...")
            continue

        feat_idx = feat_names.index(feature_name)
        arr_full = shap_d.get(method_key)
        if arr_full is None:
            arr_full = subset_shap_d.get(method_key)

        if arr_full is None:
            all_keys = list(shap_d) + [k for k in subset_shap_d if k not in shap_d]
            print(f"[{DS_LABELS[ds]}] Method key '{method_key}' not found. "
                  f"Available: {sorted(all_keys)}")
            continue

        # ── Core attribution ──────────────────────────────────────────────────
        feat_shap   = arr_full[:, feat_idx]
        mean_signed = float(feat_shap.mean())
        mean_abs    = float(np.abs(feat_shap).mean())
        is_y_parent = feat_idx in y_parent_set

        ma_all    = np.abs(arr_full).mean(axis=0)
        rank_this = int(np.argsort(np.argsort(-ma_all))[feat_idx]) + 1

        # Rank in Scratch
        rank_scratch = None
        if "Scratch" in shap_d:
            scratch_ma   = np.abs(shap_d["Scratch"]).mean(axis=0)
            rank_scratch = int(np.argsort(np.argsort(-scratch_ma))[feat_idx]) + 1

        delta_rank = (rank_this - rank_scratch) if rank_scratch is not None else None

        # ── Graph Sensitivity (GSS + SSS) ─────────────────────────────────────
        gss_val = sss_val = None
        gss_median = None
        if base_meth in BASE_METHODS:
            # Mean (RMS + output_std normalised)
            gss_map = compute_gss(shap_d, [base_meth], output_std=output_std)
            if base_meth in gss_map:
                gss_val = float(gss_map[base_meth][feat_idx])

            # Median (signed instance-level GSS)
            pc_key = f"{base_meth} (PC)"
            lg_key = f"{base_meth} (LiNGAM)"
            if pc_key in shap_d and lg_key in shap_d:
                gss_signed = np.abs(shap_d[pc_key][:, feat_idx]) - np.abs(shap_d[lg_key][:, feat_idx])
                gss_median = float(np.median(gss_signed))

            sss_map = compute_sss(shap_d, [base_meth])
            if base_meth in sss_map:
                sss_val = float(sss_map[base_meth][feat_idx])

        # ── Alignment vs True DAG ─────────────────────────────────────────────
        sa_true_val = tga_true_val = None
        tga_true_median = None
        if graph and graph not in ("Scratch",) and base_meth in BASE_METHODS:
            sa_true_map = compute_sign_alignment(
                subset_shap_d, [base_meth], [graph], reference="True")
            tga_true_map, _ = compute_tga(
                subset_shap_d, feat_names, [base_meth], [graph],
                reference="True", output_std=output_std)
            if (base_meth, graph) in sa_true_map:
                sa_true_val  = float(sa_true_map[(base_meth, graph)][feat_idx])
            if (base_meth, graph) in tga_true_map:
                tga_true_val = float(tga_true_map[(base_meth, graph)][feat_idx])

            # Median TGA vs True (signed instance-level)
            disc_key = f"{base_meth} ({graph})"
            true_key = f"{base_meth} (True)"
            if disc_key in subset_shap_d and true_key in subset_shap_d:
                tga_signed = np.abs(subset_shap_d[disc_key][:, feat_idx]) - np.abs(subset_shap_d[true_key][:, feat_idx])
                tga_true_median = float(np.median(tga_signed))

        # ── Alignment vs Scratch ───────────────────────────────────────────────
        sa_scratch_val = tga_scratch_val = None
        tga_scratch_median = None
        if graph and base_meth in BASE_METHODS:
            combined_d = {**subset_shap_d}
            if "Scratch" in shap_d:
                combined_d["Scratch"] = shap_d["Scratch"]
            sa_scr_map = compute_sign_alignment(
                combined_d, [base_meth], [graph], reference="Scratch")
            tga_scr_map, _ = compute_tga(
                combined_d, feat_names, [base_meth], [graph],
                reference="Scratch", output_std=output_std)
            if (base_meth, graph) in sa_scr_map:
                sa_scratch_val  = float(sa_scr_map[(base_meth, graph)][feat_idx])
            if (base_meth, graph) in tga_scr_map:
                tga_scratch_val = float(tga_scr_map[(base_meth, graph)][feat_idx])

            # Median TGA vs Scratch (signed instance-level)
            disc_key = f"{base_meth} ({graph})"
            if disc_key in subset_shap_d and "Scratch" in shap_d:
                tga_scr_signed = np.abs(subset_shap_d[disc_key][:, feat_idx]) - np.abs(shap_d["Scratch"][:, feat_idx])
                tga_scratch_median = float(np.median(tga_scr_signed))

        # ── Rank comparison across all loaded method keys ─────────────────────
        all_keys = list(shap_d.keys()) + [k for k in subset_shap_d if k not in shap_d]
        rank_rows = []
        for k in sorted(all_keys):
            src    = shap_d if k in shap_d else subset_shap_d
            arr_k  = src[k]
            ma_k   = np.abs(arr_k).mean(axis=0)
            ms_k   = arr_k[:, feat_idx].mean()
            rk     = int(np.argsort(np.argsort(-ma_k))[feat_idx]) + 1
            rank_rows.append({
                "Method":       k,
                "Rank":         rk,
                "Mean SHAP":    round(float(ms_k), 6),
                "Mean |SHAP|":  round(float(ma_k[feat_idx]), 6),
                "Current →":    "←" if k == method_key else "",
            })
        rank_df = (pd.DataFrame(rank_rows)
                     .sort_values("Mean |SHAP|", ascending=False)
                     .reset_index(drop=True))
        rank_df.index += 1

        # ── Print ─────────────────────────────────────────────────────────────
        print(f"\n{'═' * W}")
        print(f"  FEATURE PROFILE: {feature_name}  |  {method_key}  |  {DS_LABELS[ds]}")
        print(f"{'═' * W}")

        print(f"\n  CORE ATTRIBUTION")
        print(f"    Mean SHAP (signed)  :  {mean_signed:+.6f}")
        print(f"    Mean |SHAP|         :   {mean_abs:.6f}  →  Rank {rank_this}/{n_feat}")
        print(f"    Is Y-parent (true)  :   {'Yes ✓' if is_y_parent else 'No'}")
        if rank_scratch is not None:
            sign = "+" if delta_rank > 0 else ""
            print(f"    Rank vs Scratch     :   {rank_scratch}  →  {rank_this}"
                  f"  ({sign}{delta_rank}  {'↓ less important' if delta_rank > 0 else '↑ more important' if delta_rank < 0 else '= no change'})")

        if gss_val is not None or sss_val is not None:
            print(f"\n  GRAPH SENSITIVITY  ({base_meth}: PC vs LiNGAM)")
            if gss_val is not None:
                dir_str = "PC assigns more weight" if gss_val > 0 else "LiNGAM assigns more weight"
                print(f"    GSS (mean)         :  {gss_val:+.6f}  ({dir_str})")
                if gss_median is not None:
                    dir_str_med = "PC assigns more weight" if gss_median > 0 else "LiNGAM assigns more weight"
                    print(f"    GSS (median)       :  {gss_median:+.6f}  ({dir_str_med})")
            if sss_val is not None:
                stab = "stable" if sss_val >= 0.8 else ("unstable" if sss_val < 0.5 else "moderate")
                print(f"    SSS (sign)         :   {sss_val:.4f}  ({stab})")

        if sa_true_val is not None or tga_true_val is not None:
            print(f"\n  ALIGNMENT VS TRUE DAG  ({base_meth} — {graph})")
            if sa_true_val is not None:
                agree = "agrees" if sa_true_val >= 0.8 else ("disagrees" if sa_true_val < 0.5 else "partially agrees")
                print(f"    Sign Alignment     :   {sa_true_val:.4f}  ({agree} with true oracle)")
            if tga_true_val is not None:
                oe = "over-estimates" if tga_true_val > 0 else "under-estimates"
                print(f"    TGA (mean)         :  {tga_true_val:+.6f}  ({oe} true-graph magnitude)")
                if tga_true_median is not None:
                    oe_med = "over-estimates" if tga_true_median > 0 else "under-estimates"
                    print(f"    TGA (median)       :  {tga_true_median:+.6f}  ({oe_med} true-graph magnitude)")

        if sa_scratch_val is not None or tga_scratch_val is not None:
            print(f"\n  ALIGNMENT VS SCRATCH  ({base_meth} — {graph})")
            if sa_scratch_val is not None:
                agree = "agrees" if sa_scratch_val >= 0.8 else ("disagrees" if sa_scratch_val < 0.5 else "partially agrees")
                print(f"    Sign Alignment     :   {sa_scratch_val:.4f}  ({agree} with scratch baseline)")
            if tga_scratch_val is not None:
                oe = "over-estimates" if tga_scratch_val > 0 else "under-estimates"
                print(f"    TGA (mean)         :  {tga_scratch_val:+.6f}  ({oe} scratch magnitude)")
                if tga_scratch_median is not None:
                    oe_med = "over-estimates" if tga_scratch_median > 0 else "under-estimates"
                    print(f"    TGA (median)       :  {tga_scratch_median:+.6f}  ({oe_med} scratch magnitude)")

        print(f"\n  RANK COMPARISON  (all methods, feature: {feature_name})")
        display(rank_df)

    return rank_df


# ── Example usage ─────────────────────────────────────────────────────────────
# Replace with any feature name and method key visible in shap_data / subset_shap_data:
#
#   feature_profile("X_3",  "Asymmetric (PC)")
#   feature_profile("X_17", "Causal (LiNGAM)",  dataset="linear_conf_f50_s1000_p50")
#   feature_profile("X_42", "Scratch")


## 4. Graph Sensitivity Score (GSS)

How much does each method's output shift between the PC and LiNGAM graphs?  
Lower is more stable under graph uncertainty. GSS = 1 means the largest shift across all methods.

In [5]:
# GSS grouped bar — dataset × method
fig_gss = go.Figure()
for meth in BASE_METHODS:
    sub = df_gss[df_gss["Method"] == meth]
    fig_gss.add_trace(go.Bar(
        name         = meth,
        x            = sub["Dataset"].tolist(),
        y            = sub["MeanGSS"].tolist(),
        marker_color = METHOD_COLORS[meth],
        error_y      = None,
        hovertemplate=f"<b>{meth}</b><br>Dataset=%{{x}}<br>|GSS|=%{{y:.4f}}<extra></extra>",
    ))

fig_gss.update_layout(
    title=dict(text="Graph Sensitivity Score (Mean |GSS|) — Lower = PC and LiNGAM Agree<br>"
               "<sup>Absolute magnitude difference between PC and LiNGAM Shapley values</sup>",
               font=dict(size=13)),
    barmode="group",
    yaxis=dict(title="Mean |GSS|", zeroline=False, rangemode="tozero"),
    height=380, width=700,
    legend=dict(x=0.01, y=0.99, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=30, t=90, b=80),
    xaxis=dict(tickangle=-20),
)
fig_gss.show()

## 4b. Sign Stability Score (SSS) & Cross-Evaluation with GSS

**SSS** is the sign-domain analogue of the True-Graph Alignment sign metric: for each (instance, feature) pair we ask whether the *sign* of the SHAP value **agrees** between the PC and LiNGAM graph — consistent with how `compute_sign_alignment` compares a discovered graph against the True DAG.

$$\text{SSS}(m, f) = \frac{1}{|\{i : \phi^{PC}_{i,f} \neq 0 \;\wedge\; \phi^{LiNGAM}_{i,f} \neq 0\}|} \sum_i \mathbf{1}\!\left[\text{sign}\!\left(\phi^{PC}_{i,f}\right) = \text{sign}\!\left(\phi^{LiNGAM}_{i,f}\right)\right]$$

Higher SSS = signs are **more consistent** between the two discovered graphs (more stable).

The **cross-evaluation scatter** places every (method × dataset) in a common stability space where both axes measure instability (lower = more stable):
- **x-axis — GSS**: magnitude instability between PC and LiNGAM  
- **y-axis — (1 − SSS)**: sign flip rate between PC and LiNGAM  
- **Bottom-left corner** = maximally stable across both dimensions.

In [6]:
# ── Compute SSS via analysis_utils.compute_sss ────────────────────────────────
# Returns dict: method → ndarray(n_features,) agreement rate in [0, 1].
# Higher = signs more consistent between PC and LiNGAM graphs.

sss_metrics = []

for ds in DATASETS:
    d         = all_data[ds]
    shap_data = d["shap_data"]

    sss_per_method = compute_sss(shap_data, BASE_METHODS)

    for meth, feat_arr in sss_per_method.items():
        sss_metrics.append(dict(
            Dataset = DS_LABELS[ds],
            Method  = meth,
            MeanSSS = float(np.nanmean(feat_arr)),
        ))

df_sss = pd.DataFrame(sss_metrics)
print(df_sss.to_string(index=False))

 Dataset     Method  MeanSSS
Lin-Conf Asymmetric 0.940709
Lin-Conf     Causal 0.620600
Lin-Conf       Flow 0.867267


In [7]:
# ── SSS bar chart ─────────────────────────────────────────────────────────────
fig_sss = go.Figure()
for meth in BASE_METHODS:
    sub = df_sss[df_sss["Method"] == meth]
    fig_sss.add_trace(go.Bar(
        name         = meth,
        x            = sub["Dataset"].tolist(),
        y            = sub["MeanSSS"].tolist(),
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>Dataset=%{{x}}<br>MeanSSS=%{{y:.3f}}<extra></extra>",
    ))

fig_sss.update_layout(
    title=dict(text="Sign Stability Score (Mean SSS) — Higher = More Stable<br>"
               "<sup>Fraction of (instance, feature) pairs where SHAP sign agrees between PC and LiNGAM</sup>",
               font=dict(size=13)),
    barmode="group",
    yaxis=dict(title="Mean SSS (sign agreement)", range=[0, 1.05]),
    height=380, width=700,
    legend=dict(x=0.01, y=0.01, xanchor="left", font=dict(size=11)),
    margin=dict(l=70, r=30, t=90, b=80),
    xaxis=dict(tickangle=-20),
)
fig_sss.show()

In [8]:
# ── Cross-evaluation: |GSS| vs (1 − SSS) scatter ─────────────────────────────
# Aggregate df_gss / df_sss across all datasets before plotting
df_cross = df_gss[["Method", "MeanGSS"]].merge(
    df_sss[["Method", "MeanSSS"]], on="Method"
)
df_cross["SignFlipRate"] = 1 - df_cross["MeanSSS"]

fig_cross = plot_gss_sss_scatter(
    df_gss        = df_gss,
    df_sss        = df_sss,
    method_colors = METHOD_COLORS,
    dataset       = "Synthetic Data",
    height        = 480,
    width         = 660,
)

In [9]:
# ── Summary Table: Mean Metrics by Method (GSS vs 1-SSS) ────────────────────
df_cross_agg = df_cross.groupby("Method", as_index=False).agg({
    "MeanGSS": "mean",
    "MeanSSS": "mean",
    "SignFlipRate": "mean"
})
df_cross_agg = df_cross_agg.rename(columns={
    "MeanGSS": "Mean GSS",
    "MeanSSS": "Mean SSS",
    "SignFlipRate": "Mean Sign Flip Rate (1−SSS)"
})
df_cross_agg = df_cross_agg.sort_values("Method")

print("\nSummary: Mean Metrics PC vs LiNGAM Stability (averaged across all datasets)")
print("=" * 80)
display(df_cross_agg.round(4))


Summary: Mean Metrics PC vs LiNGAM Stability (averaged across all datasets)


,Method,Mean GSS,Mean SSS,Mean Sign Flip Rate (1−SSS)
0,Asymmetric,0.0107,0.9407,0.0593
1,Causal,0.0715,0.6206,0.3794
2,Flow,0.0606,0.8673,0.1327


## 5. True-Graph Alignment

### 5a — Sign Alignment vs True DAG  
Mean fraction of instances where the causal method's SHAP sign matches the True-graph output per feature.  
Higher = more aligned with the oracle graph.

### 5b — Magnitude TGA vs True DAG  
Mean relative magnitude deviation vs True-graph output per feature.  
Lower = closer to oracle magnitude.

In [10]:
# ── 5a: Sign Alignment — faceted heatmap dataset × method per graph ──────────
for g in DISC_GRAPHS:
    sub = df_sa[df_sa["Graph"] == g]
    pivot = sub.pivot_table(index="Method", columns="Dataset", values="MeanSignAlign")
    pivot = pivot[[DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot.columns]]

    row_labels = pivot.index.tolist()
    col_labels = pivot.columns.tolist()

    fig_sa = go.Figure(go.Heatmap(
        z            = pivot.values,
        x            = col_labels,
        y            = row_labels,
        colorscale   = "Blues",
        zmin=0.5, zmax=1.0,
        text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row] for row in pivot.values],
        texttemplate = "%{text}",
        textfont     = dict(size=12),
        colorbar     = dict(title=dict(text="Sign Align", font=dict(size=11))),
        hovertemplate=f"<b>%{{y}}</b> ({g}) — %{{x}}<br>Sign Align=%{{z:.3f}}<extra></extra>",
    ))
    fig_sa.update_layout(
        title=dict(text=f"Sign Alignment vs True DAG — {g} graph<br>"
                   "<sup>Fraction of instances where sign matches oracle</sup>",
                   font=dict(size=12)),
        height=260, width=700,
        margin=dict(l=130, r=80, t=70, b=60),
        xaxis=dict(tickfont=dict(size=11), tickangle=-15),
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig_sa.show()

In [11]:
# ── 5b: Magnitude TGA heatmap per graph ───────────────────────────────────────
for g in DISC_GRAPHS:
    sub = df_tga[df_tga["Graph"] == g]
    pivot = sub.pivot_table(index="Method", columns="Dataset", values="MeanTGA")
    pivot = pivot[[DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot.columns]]

    abs_max_tga = float(np.nanmax(np.abs(pivot.values)))

    fig_tga = go.Figure(go.Heatmap(
        z            = pivot.values,
        x            = pivot.columns.tolist(),
        y            = pivot.index.tolist(),
        colorscale   = "RdBu",
        zmid=0,
        zmin=-abs_max_tga, zmax=abs_max_tga,
        text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row] for row in pivot.values],
        texttemplate = "%{text}",
        textfont     = dict(size=12),
        colorbar     = dict(title=dict(text="Mean TGA", font=dict(size=11))),
        hovertemplate=f"<b>%{{y}}</b> ({g}) — %{{x}}<br>TGA=%{{z:.4f}}<extra></extra>",
    ))
    fig_tga.update_layout(
        title=dict(text=f"Magnitude TGA vs True DAG — {g} graph<br>"
                   "<sup>Signed: positive = disc overestimates True; negative = underestimates; white ≈ 0 = matches</sup>",
                   font=dict(size=12)),
        height=260, width=700,
        margin=dict(l=130, r=80, t=70, b=60),
        xaxis=dict(tickfont=dict(size=11), tickangle=-15),
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig_tga.show()


In [12]:
# ── 5c: Summary scatter — TGA vs Sign Disagreement (True DAG) ────────────────
# Color = Method, Shape = Graph, all datasets combined
# X-axis: TGA, Y-axis: 1 - Sign Alignment (sign disagreement rate)
GRAPH_MARKERS = {"PC": "circle", "LiNGAM": "diamond"}

fig_summary_true = plot_tga_sa_scatter(
    df_sa         = df_sa,
    df_tga        = df_tga,
    method_colors = METHOD_COLORS,
    dataset       = "All Datasets",
    reference     = "True",
    height        = 480,
    width         = 660,
)

# Create summary dataframe for next cell
df_summary_true = df_sa[["Dataset", "Method", "Graph", "MeanSignAlign"]].merge(
    df_tga[["Dataset", "Method", "Graph", "MeanTGA"]],
    on=["Dataset", "Method", "Graph"]
)

In [13]:
# ── Summary Table: Mean Metrics by Method & Graph (True DAG) ─────────────────
df_summary_true_agg = df_summary_true.groupby(["Method", "Graph"], as_index=False).agg({
    "MeanSignAlign": "mean",
    "MeanTGA": "mean"
})
df_summary_true_agg["SignDisagreement"] = 1 - df_summary_true_agg["MeanSignAlign"]
df_summary_true_agg = df_summary_true_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_true_agg = df_summary_true_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs True DAG (averaged across all datasets)")
print("=" * 80)
display(df_summary_true_agg.round(4))


Summary: Mean Metrics vs True DAG (averaged across all datasets)


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
0,Asymmetric,LiNGAM,0.9370,0.0079,0.0630
2,Causal,LiNGAM,0.6254,0.0757,0.3746
4,Flow,LiNGAM,0.5509,0.0806,0.4491
1,Asymmetric,PC,0.9412,0.0102,0.0588
3,Causal,PC,0.6840,0.0532,0.3160
5,Flow,PC,0.5774,0.0734,0.4226


## 6. Methods vs Traditional Baseline

Compares each (method × graph) against **Traditional** (graph-free SHAP) using the same sign-alignment and magnitude TGA metrics as Section 5.  
This measures how much the causal graph structure *changes* the attributions relative to the Traditional baseline, across all three graph variants (PC, LiNGAM, True).

- **Sign Alignment vs Traditional** — fraction of (instance, feature) pairs where `sign(φ_method)` matches `sign(φ_Traditional)`. Higher = output signs closer to graph-free baseline.  
- **Magnitude TGA vs Traditional** — relative magnitude deviation from Traditional. Lower = magnitudes closer to graph-free baseline.


In [14]:
# ── Metrics already computed in section 2 ─────────────────────────────────────
# Using df_sa_scratch and df_tga_scratch for visualizations

print(f"Traditional baseline metrics (computed in section 2):")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")
print("\nSign Alignment vs Traditional (sample):")
print(df_sa_scratch.pivot_table(index="Method", columns=["Graph", "Dataset"], values="MeanSignAlign").round(3).head())

Traditional baseline metrics (computed in section 2):
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows

Sign Alignment vs Traditional (sample):
Graph        LiNGAM       PC
Dataset    Lin-Conf Lin-Conf
Method                      
Asymmetric    0.942    0.949
Causal        0.661    0.647
Flow          0.623    0.634


In [15]:
# ── 6a: Sign Alignment vs Traditional — (Method, Graph) × Dataset heatmap ────
pivot_sa_scratch = df_sa_scratch.pivot_table(
    index   = ["Method", "Graph"],
    columns = "Dataset",
    values  = "MeanSignAlign",
)
pivot_sa_scratch = pivot_sa_scratch[
    [DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot_sa_scratch.columns]
]
# Sort rows: group by method, order graphs as PC → LiNGAM → True
graph_order = {"PC": 0, "LiNGAM": 1, "True": 2}
pivot_sa_scratch = pivot_sa_scratch.loc[
    sorted(pivot_sa_scratch.index, key=lambda t: (t[0], graph_order.get(t[1], 9)))
]

row_labels = [f"{m} ({g})" for m, g in pivot_sa_scratch.index]
col_labels = pivot_sa_scratch.columns.tolist()

fig_sa_scratch = go.Figure(go.Heatmap(
    z            = pivot_sa_scratch.values,
    x            = col_labels,
    y            = row_labels,
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_sa_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align<br>vs Traditional", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align vs Traditional = %{z:.3f}<extra></extra>",
))
fig_sa_scratch.update_layout(
    title=dict(
        text="Sign Alignment vs Traditional Baseline<br>"
             "<sup>Fraction of instances/features where sign agrees with graph-free output</sup>",
        font=dict(size=13),
    ),
    height=400, width=700,
    margin=dict(l=170, r=100, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11), tickangle=-15),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_sa_scratch.show()


In [16]:
# ── 6b: Magnitude TGA vs Traditional — (Method, Graph) × Dataset heatmap ─────
pivot_tga_scratch = df_tga_scratch.pivot_table(
    index   = ["Method", "Graph"],
    columns = "Dataset",
    values  = "MeanTGA",
)
pivot_tga_scratch = pivot_tga_scratch[
    [DS_LABELS[ds] for ds in DATASETS if DS_LABELS[ds] in pivot_tga_scratch.columns]
]
pivot_tga_scratch = pivot_tga_scratch.loc[
    sorted(pivot_tga_scratch.index, key=lambda t: (t[0], graph_order.get(t[1], 9)))
]

row_labels_tga = [f"{m} ({g})" for m, g in pivot_tga_scratch.index]
abs_max_scratch = float(np.nanmax(np.abs(pivot_tga_scratch.values)))

fig_tga_scratch = go.Figure(go.Heatmap(
    z            = pivot_tga_scratch.values,
    x            = pivot_tga_scratch.columns.tolist(),
    y            = row_labels_tga,
    colorscale   = "RdBu",
    zmid=0,
    zmin=-abs_max_scratch, zmax=abs_max_scratch,
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_tga_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="TGA<br>vs Traditional", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>TGA vs Traditional = %{z:.4f}<extra></extra>",
))
fig_tga_scratch.update_layout(
    title=dict(
        text="Magnitude TGA vs Traditional Baseline<br>"
             "<sup>Signed: positive = method overestimates Traditional; negative = underestimates; white ≈ 0 = matches</sup>",
        font=dict(size=13),
    ),
    height=400, width=700,
    margin=dict(l=170, r=100, t=80, b=60),
    xaxis=dict(tickfont=dict(size=11), tickangle=-15),
    yaxis=dict(tickfont=dict(size=10)),
)
fig_tga_scratch.show()


In [17]:
# ── 6c: Summary scatter — |TGA| vs Sign Disagreement (Traditional baseline) ──
# X-axis: |TGA| (absolute), Y-axis: 1 - Sign Alignment (sign disagreement rate)

fig_summary_scratch = plot_tga_sa_scatter(
    df_sa         = df_sa_scratch,
    df_tga        = df_tga_scratch,
    method_colors = METHOD_COLORS,
    dataset       = "All Datasets",
    reference     = "Traditional",
    height        = 480,
    width         = 660,
)

# Create summary dataframe for next cell
df_summary_scratch = df_sa_scratch[["Dataset", "Method", "Graph", "MeanSignAlign"]].merge(
    df_tga_scratch[["Dataset", "Method", "Graph", "MeanTGA"]],
    on=["Dataset", "Method", "Graph"]
)

In [18]:
# ── Summary Table: Mean Metrics by Method & Graph (Traditional baseline) ─────
df_summary_scratch_agg = df_summary_scratch.groupby(["Method", "Graph"], as_index=False).agg({
    "MeanSignAlign": "mean",
    "MeanTGA": "mean"
})
df_summary_scratch_agg["SignDisagreement"] = 1 - df_summary_scratch_agg["MeanSignAlign"]
df_summary_scratch_agg = df_summary_scratch_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_scratch_agg = df_summary_scratch_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs Traditional Baseline (averaged across all datasets)")
print("=" * 80)
display(df_summary_scratch_agg.round(4))


Summary: Mean Metrics vs Traditional Baseline (averaged across all datasets)


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
0,Asymmetric,LiNGAM,0.9424,0.0086,0.0576
2,Causal,LiNGAM,0.6608,0.0584,0.3392
4,Flow,LiNGAM,0.6232,0.0661,0.3768
1,Asymmetric,PC,0.9492,0.0068,0.0508
3,Causal,PC,0.6470,0.0639,0.3530
5,Flow,PC,0.6342,0.0662,0.3658


## 7. Comprehensive Summary Table

All metrics in one pivoted table — easy to copy into the thesis.

In [19]:
rows_summary = []

for ds in DATASETS:
    d             = all_data[ds]
    shap_data     = d["shap_data"]

    # Pre-compute absolute GSS via the canonical function (consistent with metric cells)
    gss_feat_ds = compute_gss(shap_data, BASE_METHODS)

    for meth in BASE_METHODS:
        for g in DISC_GRAPHS:
            key = f"{meth} ({g})"
            if key not in shap_data:
                continue

            # GSS — absolute magnitude difference (consistent with compute_gss)
            gss_arr = gss_feat_ds.get(meth)
            gss_val = float(np.nanmean(gss_arr)) if gss_arr is not None else np.nan

            # Sign Alignment
            sa_row = df_sa[(df_sa["Dataset"] == DS_LABELS[ds]) &
                           (df_sa["Method"] == meth) & (df_sa["Graph"] == g)]
            sa_val = float(sa_row["MeanSignAlign"].values[0]) if not sa_row.empty else np.nan

            # TGA
            tga_row = df_tga[(df_tga["Dataset"] == DS_LABELS[ds]) &
                             (df_tga["Method"] == meth) & (df_tga["Graph"] == g)]
            tga_val = float(tga_row["MeanTGA"].values[0]) if not tga_row.empty else np.nan

            rows_summary.append(dict(
                Dataset    = DS_LABELS[ds],
                Method     = meth,
                Graph      = g,
                GSS        = round(float(gss_val), 4) if not np.isnan(gss_val) else np.nan,
                SignAlign  = round(float(sa_val), 3)  if not np.isnan(sa_val) else np.nan,
                MagTGA     = round(float(tga_val), 4) if not np.isnan(tga_val) else np.nan,
            ))

df_summary = (
    pd.DataFrame(rows_summary)
    .sort_values(["Dataset", "Graph", "Method"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"Total rows: {len(df_summary)}")
df_summary


Total rows: 6


,Dataset,Method,Graph,GSS,SignAlign,MagTGA
0,Lin-Conf,Asymmetric,LiNGAM,0.0506,0.9370,0.0079
1,Lin-Conf,Causal,LiNGAM,0.3369,0.6250,0.0757
2,Lin-Conf,Flow,LiNGAM,0.2852,0.5510,0.0806
3,Lin-Conf,Asymmetric,PC,0.0506,0.9410,0.0102
4,Lin-Conf,Causal,PC,0.3369,0.6840,0.0532
5,Lin-Conf,Flow,PC,0.2852,0.5770,0.0734


## Section 8 — Per-Feature Diagnostic: Top-K Most Disagreeing / Most Stable Features

Surface the features at **both extremes** of each metric for targeted qualitative analysis.

### 8a–8e — Most unstable / most disagreeing features

| Sub-section | Question answered |
|-------------|-------------------|
| **8a** | Which features shift the most in *importance rank* when switching from Traditional to Method/Graph? |
| **8b** | Which features have the highest *GSS* — largest magnitude instability between PC and LiNGAM? |
| **8c** | Which features have the lowest *SSS* — most sign-unstable between PC and LiNGAM? |
| **8d** | Which features have the lowest *Sign Alignment* vs the True DAG? |
| **8e** | Which features have the highest *Magnitude TGA* vs the True DAG? |

### 8f–8i — Most stable / least-changed features

| Sub-section | Question answered |
|-------------|-------------------|
| **8f** | Which features have the *smallest rank shift* — importance barely changes when adding causal graph info? |
| **8g** | Which features have the lowest *GSS* — magnitude stays most consistent between PC and LiNGAM? |
| **8h** | Which features have the highest *SSS* — sign stays most consistent between PC and LiNGAM? |
| **8i** | Which features have the lowest *Magnitude TGA* — magnitudes closest to the True DAG oracle? |

`K_DIAG = 5` throughout (change the variable in cell 8a to adjust all sub-sections).


In [20]:
# Optional: export the summary table to CSV
out_path = BASE_DIR / "data" / "explainability" / "shapley_summary_metrics.csv"
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /Users/juanrios/Documents/master_thesis/data/explainability/shapley_summary_metrics.csv


In [31]:
details=feature_profile("X24", "Flow (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X24  |  Flow (LiNGAM)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.034597
    Mean |SHAP|         :   1.896436  →  Rank 1/50
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   2  →  1  (-1  ↑ more important)

  GRAPH SENSITIVITY  (Flow: PC vs LiNGAM)
    GSS (mean)         :  +0.089014  (PC assigns more weight)
    GSS (median)       :  -0.097966  (LiNGAM assigns more weight)
    SSS (sign)         :   0.9600  (stable)

  ALIGNMENT VS TRUE DAG  (Flow — LiNGAM)
    Sign Alignment     :   0.6500  (partially agrees with true oracle)
    TGA (mean)         :  +0.416644  (over-estimates true-graph magnitude)
    TGA (median)       :  +1.136327  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Flow — LiNGAM)
    Sign Alignment     :   0.7600  (partially agrees with scratch baseline)
  

,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (LiNGAM),1,0.1076,2.6884,
2,Causal (PC),1,0.0714,2.1916,
3,Flow (LiNGAM),1,0.0346,1.8964,←
4,Flow (PC),2,0.0306,1.8408,
5,Causal (True),1,-0.0300,1.7211,
6,Scratch,2,0.0606,1.4089,
7,Asymmetric (PC),2,0.0614,1.3969,
8,Asymmetric (True),2,0.0932,1.3848,
9,Asymmetric (LiNGAM),2,0.0853,1.3795,
10,Flow (True),7,-0.0653,0.5143,


In [22]:
K_DIAG = 5

In [23]:
# ── 8b  Top-K features by GSS (magnitude instability PC vs LiNGAM) ───────────
# Ranked by |GSS| (largest magnitude change regardless of sign).
# Signed value in the table shows direction: positive = PC > LiNGAM, negative = LiNGAM > PC.

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by |GSS|  (highest magnitude instability PC vs LiNGAM)")
    print(f"  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher")
    print(f"{'═' * 72}")
    for meth, arr in gss_feat.items():
        df_g = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="GSS")
        print(f"\n  {meth}")
        display(df_g)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by |GSS|  (highest magnitude instability PC vs LiNGAM)
  Positive GSS = PC assigns higher importance;  Negative GSS = LiNGAM higher
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,GSS
rank,,
1,X33,0.0711
2,X47,0.0465
3,X41,0.0408
4,X7,0.0398
5,X24,0.0346



  Causal


,feature,GSS
rank,,
1,X47,0.2975
2,X33,0.2915
3,X2,0.2338
4,X21,0.2139
5,X30,0.2006



  Flow


,feature,GSS
rank,,
1,X33,0.4528
2,X47,0.3632
3,X46,0.1800
4,X45,0.1727
5,X32,0.1299


In [24]:
feature_profile("X33", "Flow (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X33  |  Flow (LiNGAM)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.081169
    Mean |SHAP|         :   0.283457  →  Rank 12/50
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   1  →  12  (+11  ↓ less important)

  GRAPH SENSITIVITY  (Flow: PC vs LiNGAM)
    GSS (mean)         :  +0.452789  (PC assigns more weight)
    GSS (median)       :  +1.587342  (PC assigns more weight)
    SSS (sign)         :   0.9800  (stable)

  ALIGNMENT VS TRUE DAG  (Flow — LiNGAM)
    Sign Alignment     :   0.6869  (partially agrees with true oracle)
    TGA (mean)         :  +0.078561  (over-estimates true-graph magnitude)
    TGA (median)       :  -0.060059  (under-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Flow — LiNGAM)
    Sign Alignment     :   0.7900  (partially agrees with scratch baseline)
  

,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (PC),1,0.5116,1.9920,
2,Causal (PC),2,0.2009,1.6240,
3,Asymmetric (LiNGAM),1,0.2224,1.5124,
4,Asymmetric (True),1,0.1892,1.5117,
5,Scratch,1,0.1903,1.4683,
6,Asymmetric (PC),1,0.1671,1.4266,
7,Causal (LiNGAM),5,0.0597,0.8998,
8,Causal (True),8,0.0141,0.5864,
9,Flow (True),16,-0.0052,0.3479,
10,Flow (LiNGAM),12,0.0812,0.2835,←


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (PC),1,0.5116,1.9920,
2,Causal (PC),2,0.2009,1.6240,
3,Asymmetric (LiNGAM),1,0.2224,1.5124,
4,Asymmetric (True),1,0.1892,1.5117,
5,Scratch,1,0.1903,1.4683,
6,Asymmetric (PC),1,0.1671,1.4266,
7,Causal (LiNGAM),5,0.0597,0.8998,
8,Causal (True),8,0.0141,0.5864,
9,Flow (True),16,-0.0052,0.3479,
10,Flow (LiNGAM),12,0.0812,0.2835,←


In [25]:
# ── 8c  Bottom-K features by SSS (sign instability between PC and LiNGAM) ────
# Lower SSS = that feature flips sign more often when you swap PC for LiNGAM.

for ds in DATASETS:
    label      = DS_LABELS[ds]
    shap_d     = all_data[ds]["shap_data"]
    feat_names = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by SSS  (lower = more sign-unstable)")
    print(f"{'═' * 72}")
    for meth, arr in sss_feat.items():
        df_s = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="SSS")
        print(f"\n  {meth}")
        display(df_s)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by SSS  (lower = more sign-unstable)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,SSS
rank,,
1,X13,0.7200
2,X29,0.8100
3,X3,0.8163
4,X9,0.8200
5,X34,0.8300



  Causal


,feature,SSS
rank,,
1,X8,0.3600
2,X32,0.3700
3,X10,0.4200
4,X30,0.4300
5,X11,0.4500



  Flow


,feature,SSS
rank,,
1,X10,0.4343
2,X5,0.5300
3,X26,0.5341
4,X13,0.5875
5,X43,0.6615


In [26]:
# ── 8d  Bottom-K features by Sign Alignment vs True DAG ──────────────────────
# Lowest sign alignment = the method's directional attribution most often disagrees
# with the oracle (True DAG) for that feature.

for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]
    # sa_feat     = compute_sign_alignment(sub_shap_d, BASE_METHODS, DISC_GRAPHS, reference="True")

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Bottom-{K_DIAG} features by Sign Alignment vs True DAG")
    print(f"  (lower = method/graph attributions most often flip sign vs True-DAG oracle)")
    print(f"{'═' * 72}")
    for (meth, g), arr in sign_align.items():
        df_sa_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=True, score_col="SignAlign")
        print(f"\n  {meth} ({g})")
        display(df_sa_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Bottom-5 features by Sign Alignment vs True DAG
  (lower = method/graph attributions most often flip sign vs True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,SignAlign
rank,,
1,X27,0.5700
2,X13,0.7200
3,X29,0.8100
4,X3,0.8100
5,X9,0.8100



  Asymmetric (LiNGAM)


,feature,SignAlign
rank,,
1,X27,0.5900
2,X9,0.6300
3,X3,0.7300
4,X34,0.7700
5,X10,0.8500



  Causal (PC)


...

,feature,SignAlign
rank,,
1,X42,0.3500
2,X32,0.3800
3,X36,0.4000
4,X10,0.4100
5,X30,0.4200



  Flow (PC)


,feature,SignAlign
rank,,
1,X27,0.2184
2,X36,0.2727
3,X35,0.3700
4,X34,0.3737
5,X8,0.3776



  Flow (LiNGAM)


,feature,SignAlign
rank,,
1,X27,0.2069
2,X36,0.2828
3,X14,0.3030
4,X16,0.3200
5,X8,0.3571


In [27]:
# ── 8e  Top-K features by Magnitude TGA vs True DAG ──────────────────────────
# Highest TGA = the method's attribution magnitudes deviate the most from the
# oracle (True DAG) for that feature; TGA > 1 means the error exceeds the reference mean.
K_DIAG = 5
for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by Magnitude TGA vs True DAG")
    print(f"  (higher = absolute magnitudes furthest from the True-DAG oracle;")
    print(f"   TGA > 1 means error exceeds the oracle's own mean attribution)")
    print(f"{'═' * 72}")
    for (meth, g), arr in tga_data.items():
        df_tga_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="TGA")
        print(f"\n  {meth} ({g})")
        display(df_tga_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by Magnitude TGA vs True DAG
  (higher = absolute magnitudes furthest from the True-DAG oracle;
   TGA > 1 means error exceeds the oracle's own mean attribution)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,X33,0.0765
2,X24,0.0475
3,X47,0.0430
4,X46,0.0316
5,X7,0.0293



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,X6,0.0367
2,X24,0.0326
3,X33,0.0293
4,X41,0.0204
5,X40,0.0201



  Causal (PC)


...

,feature,TGA
rank,,
1,X47,0.2940
2,X6,0.2810
3,X24,0.2692
4,X21,0.2092
5,X45,0.2000



  Flow (PC)


,feature,TGA
rank,,
1,X33,0.4701
2,X24,0.4062
3,X44,0.1796
4,X45,0.1475
5,X21,0.1347



  Flow (LiNGAM)


,feature,TGA
rank,,
1,X24,0.4166
2,X47,0.4111
3,X46,0.2523
4,X44,0.1819
5,X48,0.1804


In [28]:
feature_profile("X47", "Asymmetric (PC)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X47  |  Asymmetric (PC)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.002461
    Mean |SHAP|         :   1.111062  →  Rank 3/50
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   3  →  3  (0  = no change)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (mean)         :  +0.046477  (PC assigns more weight)
    GSS (median)       :  +0.055586  (PC assigns more weight)
    SSS (sign)         :   0.8900  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — PC)
    Sign Alignment     :   0.9200  (agrees with true oracle)
    TGA (mean)         :  +0.042994  (over-estimates true-graph magnitude)
    TGA (median)       :  +0.043816  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — PC)
    Sign Alignment     :   0.9700  (agrees with scratch baseline)
    TGA (mean)      

,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (LiNGAM),2,-0.2015,1.6714,
2,Causal (LiNGAM),2,0.0015,1.4286,
3,Asymmetric (PC),3,0.0025,1.1111,←
4,Asymmetric (True),3,0.0291,1.0961,
5,Scratch,3,0.0154,1.0831,
6,Asymmetric (LiNGAM),3,0.0265,1.0758,
7,Causal (True),10,-0.0929,0.5284,
8,Causal (PC),8,-0.0925,0.5224,
9,Flow (PC),13,-0.0484,0.3821,
10,Flow (True),14,-0.0369,0.3619,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (LiNGAM),2,-0.2015,1.6714,
2,Causal (LiNGAM),2,0.0015,1.4286,
3,Asymmetric (PC),3,0.0025,1.1111,←
4,Asymmetric (True),3,0.0291,1.0961,
5,Scratch,3,0.0154,1.0831,
6,Asymmetric (LiNGAM),3,0.0265,1.0758,
7,Causal (True),10,-0.0929,0.5284,
8,Causal (PC),8,-0.0925,0.5224,
9,Flow (PC),13,-0.0484,0.3821,
10,Flow (True),14,-0.0369,0.3619,


In [29]:
# ── 8f  Top-K features by Magnitude TGA vs Traditional DAG ──────────────────────────
# Highest TGA = the method's attribution magnitudes deviate the most from the
# oracle (Traditional DAG) for that feature; TGA > 1 means the error exceeds the reference mean.
K_DIAG = 5
for ds in DATASETS:
    label       = DS_LABELS[ds]
    sub_shap_d  = all_data[ds]["subset_shap_data"]
    feat_names  = all_data[ds]["feature_names"]

    print(f"\n{'═' * 72}")
    print(f"  {label}  ·  Top-{K_DIAG} features by Magnitude TGA vs Traditional DAG")
    print(f"  (higher = absolute magnitudes furthest from the Traditional shapley value;")
    print(f"   TGA > 1 means error exceeds the oracle's own mean attribution)")
    print(f"{'═' * 72}")
    for (meth, g), arr in tga_scratch.items():
        df_tga_f = top_k_features(arr, feat_names, k=K_DIAG, ascending=False, score_col="TGA")
        print(f"\n  {meth} ({g})")
        display(df_tga_f)



════════════════════════════════════════════════════════════════════════
  Lin-Conf  ·  Top-5 features by Magnitude TGA vs Traditional DAG
  (higher = absolute magnitudes furthest from the Traditional shapley value;
   TGA > 1 means error exceeds the oracle's own mean attribution)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,X33,0.0336
2,X47,0.0281
3,X41,0.0249
4,X24,0.0220
5,X7,0.0206



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,X33,0.0469
2,X24,0.0351
3,X41,0.0267
4,X46,0.0260
5,X6,0.0251



  Causal (PC)


...

,feature,TGA
rank,,
1,X24,0.3280
2,X2,0.2688
3,X33,0.2614
4,X7,0.1512
5,X41,0.1334



  Flow (PC)


,feature,TGA
rank,,
1,X24,0.3206
2,X33,0.3036
3,X47,0.2059
4,X45,0.1737
5,X21,0.1603



  Flow (LiNGAM)


,feature,TGA
rank,,
1,X24,0.3246
2,X33,0.3173
3,X47,0.3005
4,X46,0.2117
5,X7,0.1457


In [30]:
feature_profile("X6", "Asymmetric (PC)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: X6  |  Asymmetric (PC)  |  Lin-Conf
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.041901
    Mean |SHAP|         :   0.461368  →  Rank 10/50
    Is Y-parent (true)  :   No
    Rank vs Scratch     :   10  →  10  (0  = no change)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (mean)         :  +0.022061  (PC assigns more weight)
    GSS (median)       :  +0.002452  (PC assigns more weight)
    SSS (sign)         :   0.9900  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — PC)
    Sign Alignment     :   1.0000  (agrees with true oracle)
    TGA (mean)         :  +0.028156  (over-estimates true-graph magnitude)
    TGA (median)       :  -0.019751  (under-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — PC)
    Sign Alignment     :   1.0000  (agrees with scratch baseline)
    TGA (mean)      

,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (True),2,0.0506,1.1328,
2,Asymmetric (PC),10,0.0419,0.4614,←
3,Asymmetric (LiNGAM),10,0.0419,0.4573,
4,Scratch,10,0.0586,0.4564,
5,Asymmetric (True),10,0.0542,0.4459,
6,Flow (True),10,0.0030,0.4195,
7,Causal (PC),14,0.0794,0.3879,
8,Flow (PC),25,0.0418,0.1642,
9,Causal (LiNGAM),26,0.0121,0.1365,
10,Flow (LiNGAM),31,0.0148,0.0634,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (True),2,0.0506,1.1328,
2,Asymmetric (PC),10,0.0419,0.4614,←
3,Asymmetric (LiNGAM),10,0.0419,0.4573,
4,Scratch,10,0.0586,0.4564,
5,Asymmetric (True),10,0.0542,0.4459,
6,Flow (True),10,0.0030,0.4195,
7,Causal (PC),14,0.0794,0.3879,
8,Flow (PC),25,0.0418,0.1642,
9,Causal (LiNGAM),26,0.0121,0.1365,
10,Flow (LiNGAM),31,0.0148,0.0634,
